# document extractor

In [2]:
# https://onlyoneaman.medium.com/i-tested-7-python-pdf-extractors-so-you-dont-have-to-2025-edition-c88013922257

import textract
text = textract.process(r"/Users/karthickthangadurai/PersonalPro/My_Projects/Otelio/data/hotel_rag_document_v2.pdf").decode()

In [1]:
print(text)

NameError: name 'text' is not defined

In [6]:
# pip install "unstructured[all-docs]"
from unstructured.partition.auto import partition
blocks = partition(filename="data/hotel_rag_document_v2.pdf")
for block in blocks:
    print(f"{block.category}: {block.text}")

Title: Grand Azure Bay Hotel - Detailed Information Guide
Title: Overview
NarrativeText: Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.
Title: Location & Accessibility
NarrativeText: The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.
Title: Frequently Asked Questions - General
NarrativeText: Check-in time is 2:00 PM and check-out time is 11:00 AM. Early check-in and late check-out are subject to availability. The hotel provides free Wi-Fi, complimentary breakfast for select bookings, and 24/7 customer support.
Title: Hy

In [3]:
"""Evaluated PyMuPDF, textract, and unstructured; chose unstructured because its element-type 
classification (Title/NarrativeText) enables deterministic section-based chunking, whereas the plain-text extractors 
collapse headings into body text and would require brittle heuristics."""


from unstructured.partition.pdf import partition_pdf

blocks = partition_pdf(
    filename="data/hotel_rag_document_v2.pdf",
    strategy="fast"
)

In [4]:
for block in blocks:
    print(f"{block.category}: {block.text}")

Title: Grand Azure Bay Hotel - Detailed Information Guide
Title: Overview
NarrativeText: Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.
Title: Location & Accessibility
NarrativeText: The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.
Title: Frequently Asked Questions - General
NarrativeText: Check-in time is 2:00 PM and check-out time is 11:00 AM. Early check-in and late check-out are subject to availability. The hotel provides free Wi-Fi, complimentary breakfast for select bookings, and 24/7 customer support.
Title: Hy

In [ ]:
import pymupdf

doc = pymupdf.open("hotel_rag_document_v2.pdf")
for page in doc:
    print(page.get_text())

Grand Azure Bay Hotel - Detailed Information
Guide
Overview
Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine
dining, and personalized guest experiences. The hotel is designed to cater to both business and
leisure travelers with a focus on comfort, safety, and service excellence.
Location & Accessibility
The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from
the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle
services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife
hubs.
Frequently Asked Questions - General
Check-in time is 2:00 PM and check-out time is 11:00 AM. Early check-in and late check-out are
subject to availability. The hotel provides free Wi-Fi, complimentary breakfast for select bookings,
and 24/7 customer support.

Hygiene & Cleanliness Protocols
The hotel follows strict hygiene standards aligne

# Chunking

In [5]:
from unstructured.chunking.title import chunk_by_title

chunks = chunk_by_title(blocks, combine_text_under_n_chars=0, include_orig_elements = True)

for chunk in chunks:
    print(chunk)
    print("\n\n" + "-"*80)

Grand Azure Bay Hotel - Detailed Information Guide


--------------------------------------------------------------------------------
Overview

Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.


--------------------------------------------------------------------------------
Location & Accessibility

The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.


--------------------------------------------------------------------------------
Frequently Asked Questions - General

Check-in time is 2:00 PM and check-out time is 11:00 

In [6]:
"""
Chunks carry a hotel metadata field; single-property today, but the schema and retrieval filter 
extend to multi-property without re-ingestion redesign.
"""

print(chunks[2].text)

Location & Accessibility

The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.


In [11]:
print(chunks[2].metadata.to_dict())

{'file_directory': 'data', 'filename': 'hotel_rag_document_v2.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2026-07-18T20:06:46', 'page_number': 1, 'orig_elements': 'eJzdU01v2zAM/SuEDzulnuPYjtzbUOwwYCgGLLegCGiJjonKkiHJWYKi/32S02Ldx2WXHXYk3yP1+EjtnzLSNJIJB1bZLWStwG0r13WNXVeJTa0kdVSWnewrIftWZCvIRgqoMGDkP2XSWqfYYCC/xBovdg6Hgfg4hJgR1ToXohWx7gX6xioMEanbOi+3dRORybIJqX6/34q8WEFZF3n7sILXsKmuYblpc/GHeKHHROYvPtCYJvnCZ9JfJ5SUPUegZ00HxY5ksO6SCMsIL4jBkVJusIH0weHxoKycF19OZT6p/pUYLtNCxGnSLDGwNe9fYI3mOONx8WGfkTlmD0vWh8NoFfdMi8NlUTY3xfZmLXZlcVs0t1WTqqdYeTDz2JGLrHWSHOicHMw+2+tD8A4+SEnec8eawyWVverZcdDLoL8utBMoGllto1MNtqJuKiRRUF8UqiQhy3+1UCHy9Y+FbjbVNazX67z+Pb7S/6eFLhn3Fx/t7QXsBoJlEmAPOp0DKWADH/XFMxq4s1EU3MWbWEEcxNkzj5GjL1DD4wi9syOE2ENGBsgoghygUVAWP8GcALN4gBqQ3WRdyOFTSM8Seo4N8eUCNcGJEQKeeQXT3EXzIDg0PtWslu6L4ht0MXuMev0wh3ik4MmdODbJ4Z7QdbFliIUyveqjBKlnRdARyoH8CuSsw+yinLgNNaJ7jDk/2GlicwTFPjiWwV8fNOlENffRrLnz+dv/cZ9kBD7RLln6/PAdQzaR9Q=='}


In [ ]:
from unstructured.staging.base import elements_from_base64_gzipped_json

records = []

hotel_id = str(chunks[0]).split('-')[0].strip().lower().replace(" ", "-")
hotel_name = str(chunks[0]).split('-')[0].strip()

for element, counter in zip(chunks[1:], range(1, len(chunks[1:]) + 1)):

    
    metadata = element.metadata.to_dict()

    id = f"{hotel_id}-{counter}"
    page_number = metadata["page_number"]
    title = ''
    
    content = ''

    orig_elements = elements_from_base64_gzipped_json(metadata["orig_elements"])

    for orig_element in orig_elements:
        
        content += orig_element.text + " : "

        if orig_element.category == "Title":

            title = orig_element.text

    doc = {
        'metadata': {
            'id': id,
            'hotel_name': hotel_name,
            'page_number': page_number,
            'title': title
        },
        'content': content
    }

    records.append(doc)

print(records[0])

{'metadata': {'id': 'grand-azure-bay-hotel-1', 'hotel_name': 'Grand Azure Bay Hotel', 'page_number': 1, 'title': 'Overview'}, 'content': 'OverviewGrand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.'}


In [7]:
from unstructured.staging.base import elements_from_base64_gzipped_json

records = []

# hotel name from the document title (first chunk), split on " - " 
doc_title = str(chunks[0]).strip()
hotel_name = doc_title.split(" - ")[0].strip()          # "Grand Azure Bay Hotel"
hotel_slug = hotel_name.lower().replace(" ", "-")       # "grand-azure-bay-hotel"

for counter, element in enumerate(chunks[1:], start=1):
    metadata = element.metadata.to_dict()
    orig_elements = elements_from_base64_gzipped_json(metadata["orig_elements"])

    title = "General"
    parts = []
    for e in orig_elements:
        if e.category == "Title":
            title = e.text
        else:
            parts.append(e.text)
    narrative = " ".join(parts)

    if len(narrative) < 30:        # skip title-only / junk chunks
        continue

    records.append({
        "id": f"{hotel_slug}-{counter}",
        "content": f"{title}: {narrative}",
        "metadata": {
            "hotel_name": hotel_name,
            "section": title,
            "page_number": metadata.get("page_number", 1),
            "chunk_id": counter,
        },
    })

print(len(records))                 # expect 16
print(records[0]["content"][:100])  # expect "Overview: Grand Azure Bay Hotel is a luxury..."

15
Overview: Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine


In [8]:
records

[{'id': 'grand-azure-bay-hotel-1',
  'content': 'Overview: Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.',
  'metadata': {'hotel_name': 'Grand Azure Bay Hotel',
   'section': 'Overview',
   'page_number': 1,
   'chunk_id': 1}},
 {'id': 'grand-azure-bay-hotel-2',
  'content': 'Location & Accessibility: The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.',
  'metadata': {'hotel_name': 'Grand Azure Bay Hotel',
   'section': 'Location & Accessibility',
   'page_number': 1,
   'chunk_id': 2}},
 {'id': 'grand-azure-bay-hotel-

In [10]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

client = chromadb.PersistentClient(path="./chroma_db")
ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.get_or_create_collection(
    name="hotel_docs",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

collection.upsert(
    ids=[r["id"] for r in records],
    documents=[r["content"] for r in records],
    metadatas=[r["metadata"] for r in records],
)
print(f"Loaded {collection.count()} chunks")

# ---- the four-query retrieval test ----
for q in [
    "What is the famous dish in the hotel?",
    "How does the hotel ensure hygiene?",
    "Is vegetarian food available?",
    "What is the cancellation policy?",
]:
    res = collection.query(query_texts=[q], n_results=3)
    print(q, "->", [m["section"] for m in res["metadatas"][0]])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 15 chunks
What is the famous dish in the hotel? -> ['Dining Experience', 'Famous Dishes', 'Common User Questions - Food']
How does the hotel ensure hygiene? -> ['Hygiene & Cleanliness Protocols', 'Common User Questions - Safety', 'Guest Experience & Services']
Is vegetarian food available? -> ['Common User Questions - Food', 'Famous Dishes', 'Food Safety & Kitchen Standards']
What is the cancellation policy? -> ['Cancellation & Modification Policy', 'Common User Questions - Booking', 'Reservation Process']
